# Banking clustering: group similar loan applicants
Clusters are **discovered groups**, not correct/incorrect labels. The loan-default outcome must not enter the clustering features.
# Installation and data
Use a Python environment supported by your installed PyCaret release; check the release installation page before installing. In a fresh Python 3.11 environment, run `%pip install pycaret` and restart the kernel. Keep the named CSV alongside this notebook. These notebooks use PyCaret 3 function APIs and the data is for teaching only.


## 1. Read and select comparable numeric features
Keep meaningful applicant characteristics and remove identifiers and outcome labels.

In [ ]:
import pandas as pd
df = pd.read_csv("Banking_Loan_Default_Classification.csv")
print(df.shape)
display(df.head())
display(df.isna().sum().to_frame("missing"))
features = ["age", "monthly_income_inr", "years_employed", "loan_amount_inr", "monthly_debt_payments_inr", "credit_score", "late_payments_last_12m", "savings_balance_inr"]
X = df[features].copy()
display(X.describe().T)

## 2. Initialize clustering
Normalization makes income, savings and credit score comparable in distance calculations.

In [ ]:
from pycaret.clustering import setup, models, create_model, pull, assign_model, predict_model, save_model, load_model
exp = setup(data=X, normalize=True, session_id=42, verbose=False)
display(models())

## 3. Create two clusterings
K-means needs a chosen number of clusters. Silhouette measures separation from nearby groups; larger is usually better, but also inspect segment meaning.

In [ ]:
k3 = create_model("kmeans", num_clusters=3)
display(pull())
k4 = create_model("kmeans", num_clusters=4)
display(pull())

## 4. Interpret the segments
Inspect group sizes and original-scale means. Cluster numbers are arbitrary labels, and no accuracy or confusion matrix applies.

In [ ]:
assigned = assign_model(k3)
display(assigned["Cluster"].value_counts())
display(assigned.groupby("Cluster")[features].mean().round(1))

## 5. Save, reload and assign new applicants
A stored pipeline applies the same transformations to new rows.

In [ ]:
save_model(k3, "banking_customer_segments")
loaded = load_model("banking_customer_segments")
display(predict_model(loaded, data=X.head(5)))